In [ ]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from selenium import webdriver

import pandas as pd

from time import sleep

import datetime

from bs4 import BeautifulSoup

from pandas import ExcelWriter

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC

import os

from selenium.webdriver.chrome.service import Service as ChromeService



# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'PT BPOR' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.3")

now=datetime.datetime.now()

filename = '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

#scriptfolder = f"C:\\Users\\siewekoa\\OneDrive - moodys.com\\Desktop\\My_data\\Project_work\\scripts_regulator\\{regulatorName}" ## to comment for the production environment

scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_argument("--disable-search-engine-choice-screen")

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

        'PT BPOR 29': 'https://www.bportugal.pt/en/entidades-autorizadas',

		}



Typology={

        'PT BPOR 29': 'Authorised institutions',

		}



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}



label_row = ['Type of institution', 'Institution Acronym', 'Abbreviation', 'Status', 'Address', 'Foreign Address', 'Location', 'Foreign Location', 'Postal Code', 'Foreign Postal Code', 'Country', 'Foreign Country', 'Start date of activity', 'Name', 'IF Code' ]

# label_row_PT = ['Tipo', 'Sigla da Instituição', 'Estado', 'Morada', 'Morada Estrangeiro', 'Localidade', 'Localidade Estrangeiro', 'Código Postal', 'Código Postal Estrangeiro', 'País', 'País estrangeiro', 'Data de Início de Atividade', 'Denominação da Sede', 'Código de IF' ]

label_sqldict = ['Typology', 'InternalID_2', 'InternalID_2', 'RegulationType', 'Address_1', 'Address_1', 'City', 'City', 'Zip', 'Zip', 'Cntry', 'Cntry', 'RegulationDate', 'Name', 'InternalID_1' ]



processdate = now.strftime('%Y-%m-%d')



# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def click_on_cookies(Msg_accept):

    try:

        driver.find_element(By.XPATH,f'//*[@id="onetrust-accept-btn-handler"]').click()

        print(f'[INFO] : click "{Msg_accept}" button')

        sleep(1)

    except Exception as err:

        print(f'[ERROR] : Failed to click "{Msg_accept}" button on the cookies banner:\n {err}')





# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):

    

    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_")

    driver.get(regdict[reg])

    sleep(5)

    soup=BeautifulSoup(driver.page_source, 'html.parser')

    sleep(1)

    try :

        cookie_bannier = driver.find_element(By.XPATH, '//*[@id="onetrust-banner-sdk"]')

        cookie_is_displayed = cookie_bannier.is_displayed()

    except:

        cookie_is_displayed = False

    if cookie_is_displayed :

        click_on_cookies("Accept All Cookies")



    items=soup.find("div", {"class":"list-items-indexed--count bdpsi-label-small-highcase"})

    items = items.text.split()[0]

    # items = '30' # for test

    total_company = 0

    scrollTo = 0



    container=soup.find("div", {"class":"views-infinite-scroll-content-wrapper clearfix row row--unformatted-list"})

    all_company = container.find_all('a')

    preview_total_company = len(all_company)

    links = ['https://www.bportugal.pt'+link['href'] for link in all_company]



    while total_company < int(items) :

        for k in range(20):

            driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.DOWN)

            sleep(0.2)

        sleep(15)

        soup=BeautifulSoup(driver.page_source, 'html.parser')

        container=soup.find("div", {"class":"views-infinite-scroll-content-wrapper clearfix row row--unformatted-list"})

        try:

            all_company = container.find_all('a')

        except:

            scrollTo = 0

            links = []

            driver.get(regdict[reg])

            print('[ERROR] : ** RELOAD WEB SITE **')

            sleep(5)

            soup=BeautifulSoup(driver.page_source, 'html.parser')

            continue

        total_company = len(all_company)

        links = links + ['https://www.bportugal.pt'+link['href'] for link in all_company][preview_total_company:]

        print(f'[INFO] : Total_company = {total_company}/{int(items)} | links = {len(links)}')





        if preview_total_company == total_company :

            print(f'[INFO] : ** | preview_total_company = {preview_total_company} Total_company = {total_company} | links = {len(links)}')

            # driver.get(regdict[reg])

            # sleep(2)

            for k in range(10):

                driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.DOWN)

                sleep(0.2)

            sleep(15)

            soup=BeautifulSoup(driver.page_source, 'html.parser')

            sleep(0.5)

            container=soup.find("div", {"class":"views-infinite-scroll-content-wrapper clearfix row row--unformatted-list"})

            try:

                all_company = container.find_all('a')

            except:

                scrollTo = 0

                links = []

                driver.get(regdict[reg])

                print('[ERROR] : ** RELOAD WEB SITE **')

                sleep(5)

                soup=BeautifulSoup(driver.page_source, 'html.parser')

                continue

                

            preview_total_company = len(all_company)

            links = ['https://www.bportugal.pt'+link['href'] for link in all_company]



        # print(f'[INFO] : -- ACTUEL DATA | preview_total_company = {preview_total_company} Total_company = {total_company} | links = {len(links)} | scroll = {scrollTo}')

        preview_total_company = len(all_company)

        scrollTo +=1

        if scrollTo > 70: #52 

            raise Exception(f'[ERROR] : exit the while loop | scrollTo = {scrollTo} | total_company = {total_company}/{int(items)}' )

    

    for i, link in enumerate(links):

        values = {}

        # driver.get(link.replace('.pt/en/', '.pt/')) # for portugais version of the page

        driver.get(link)

        print(f'[INFO] : Scrapping data links = {i+1}/{len(links)}')

        sleep(2)

        soup=BeautifulSoup(driver.page_source, 'html.parser')

        main_content = soup.find('div', {"class":"row bdpsi--paragraph--row"})

        rows = main_content.find_all('div', {"class":"col-12 col-lg-4"})



        for j, row in enumerate(rows):

            label = row.find('div').text

            if label in label_row :

                try:

                    values[label_sqldict[label_row.index(label)]]=row.find_all('div')[1].text.strip()

                except:

                    values[label_sqldict[label_row.index(label)]]=rows[j].text.strip()

        

        

        # print(f'[INFO] : -- {i+1} : rows = {len(rows)} | {len(values)}')

        if 'Name' in values :

            sqldict['Name'].append(values['Name'])

            sqldict['InternalID_1'].append(values['InternalID_1']) if 'InternalID_1' in values else sqldict['InternalID_1'].append('')

            sqldict['InternalID_1_type'].append('IF Code') if 'InternalID_1' in values else sqldict['InternalID_1_type'].append('')

            sqldict['InternalID_2'].append(values['InternalID_2']) if 'InternalID_2' in values else sqldict['InternalID_2'].append('')

            sqldict['InternalID_2_type'].append('Abbreviation') if 'InternalID_2' in values else sqldict['InternalID_2_type'].append('')



            sqldict['City'].append(values['City']) if 'City' in values else sqldict['City'].append('')

            sqldict['Typology'].append(values['Typology']) if 'Typology' in values else sqldict['Typology'].append('')

            sqldict['RegulationType'].append(values['RegulationType']) if 'RegulationType' in values else sqldict['RegulationType'].append('')

            sqldict['Zip'].append(values['Zip']) if 'Zip' in values else sqldict['Zip'].append('')

            sqldict['Cntry'].append(values['Cntry']) if 'Cntry' in values else sqldict['Cntry'].append('')

            sqldict['RegulationDate'].append(values['RegulationDate']) if 'RegulationDate' in values else sqldict['RegulationDate'].append('')



            # sqldict['Typology'].append(Typology[reg])

            sqldict['ListProcessDate'].append(processdate)

            sqldict['RegCtry'].append(reg.split(' ')[0]) 

            sqldict['RegCode'].append(reg.split(' ')[1])

            sqldict['ListCode'].append(reg.split(' ')[-1]) 

            

            sqldict = bourange_same_length_array(sqldict)



        # if i == 5 : # for test

        #     break





# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

#Moving the file to the output folder (this way it will be displayed in the Control Room)

sleep(3)
    
    